In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI  # Added for Google Gemini (AI Studio)
from langchain.agents import create_agent  # New in LangChain v1.x: replaces create_react_agent & AgentExecutor
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langsmith import Client  # New replacement for `from langchain import hub` (use Client().pull_prompt(...))


In [5]:
#from langchain.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent

In [6]:
#import env
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [10]:
search_tool = TavilySearchResults(max_results=3)

In [11]:
result = search_tool.invoke("give me latest news about AI")

In [12]:
result

[{'title': 'Artificial intelligence',
  'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
  'content': 'GPT-3 is a large language model that was released in 2020 by OpenAI and is capable of generating high-quality human-like text.( launched on 30 November 2022, became the fastest-growing consumer software application in history, gaining over 100 million users in two months.( It marked what is widely regarded_[weasel words_] as AI\'s breakout year, bringing it into the public consciousness.( source needed_] These programs, and others, lead to an AI boom, where large companies began investing billions of dollars in AI research. According to AI Impacts,_[who?_] about US$50 billion annually was invested in "AI" around 2022 in the U.S. alone and about 20% of the new U.S. computer science PhD graduates have specialised in "AI".( About 800,000 "AI"-related U.S. job openings existed in 2022.( [...] ### Web search\n\nMicrosoft introduced Copilot Search in February 2023 under the n

In [13]:
#initializing the llm
llm = ChatOpenAI(
    model_name="gpt-6-luna",
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

In [ ]:
response = llm.invoke("what is the latest news about nvidia ceo jensen huang?")
response

#check if the llm is rate limited
# from openai import RateLimitError

# try:
#     response = llm.invoke("what is the latest news about nvidia ceo")
# except RateLimitError as e:
#     print(e.body)

{'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}


In [ ]:
# run this commmand install the required packages bcz some packages names are chaged in the latest version of langchain
#pip install -U langchain langchain-core langchain-community langchain-openai langchain-google-genai langsmith python-dotenv

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

response = llm.invoke("what is the latest news about nvidia ceo")
print(response.content)




Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text', 'text': 'As of recent months, NVIDIA CEO **Jensen Huang** has remained at the center of the global technology and financial spotlight. Here is a summary of the latest and most significant news surrounding him:\n\n---\n\n### 1. High Demand and Production of the "Blackwell" Chips\n* **"Insane" Demand:** Huang has repeatedly stated that demand for NVIDIA’s next-generation **Blackwell AI chips** is "insane," with tech giants like Microsoft, Meta, Google, and OpenAI racing to secure capacity.\n* **Overcoming Delays:** Huang recently addressed reports regarding Blackwell production delays, confirming there was an initial design flaw related to packaging that reduced yields. He clarified that NVIDIA and manufacturing partner **TSMC** resolved the issue and the chips are in full mass production.\n\n---\n\n### 2. Major Stock Sales\n* Throughout mid-to-late 2024, Huang made headlines for selling a significant portion of his personal NVIDIA shares under a pre-arranged **Rule 10b

In [23]:
#prompt = hub.pull("hwchase17/react")
from langsmith import Client

prompt = Client().pull_prompt(
    "hwchase17/react", dangerously_pull_public_prompt=True
)
print(prompt)


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [24]:
tools = [search_tool]


In [37]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.template,
    debug=True
)

In [38]:
res = agent.invoke({
    "messages": [{"role": "user", "content": "find the capital of india"}]
})


[values] {'messages': [HumanMessage(content='find the capital of india', additional_kwargs={}, response_metadata={}, id='7e485fb7-2a49-4fdc-aef5-1f881fc790df')]}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Thought: The capital of India is well-known to be New Delhi.\nFinal Answer: The capital of India is New Delhi.', 'extras': {'signature': 'EuoDCucDAWkUfRNpQoAM5KtLXx0xzP+2o++LK5MytvxR45aPtP4Qk7tDSQgw3NVcd6rbHq6dVK+16hUCXoBRnldsYbE4WOpTupCqdRN3HUhARN0qna71pRty3HAKu+TS/T68Hp45H5Jpw8Vlo/nRK24JrAKEP85n+uqnZx2XvXDcOegzvTTL6ww0xmgDbDUCYx8Se5NyPZ2+CyCuBN2A1wjdQXKLvduifGt3YLzbDyY4zisMNpP1B7vGdbd24E/5dNq9ngOZOVN8fT9KUTbuZd+ZKr7bYoY97I8vZoR2tC2wKnDl90OBgK1vvezZ/ZRMs/kPfBBZWNBxWZhPPC5fR8zMb6kJDdxzckYSm9LLi4uINdoaENBOgHa3KCorkhspHj8c8Q99usE/7Nxs9nIlpekTbzsfk6APJzZ+FzdRUfVYwU/dqClHOfEV022ySW2IAfbBF3dEQCWFaq/mdDmELZcoseJHAaV+jMbk/0aYgV3mCfgSpXdWbgIH6i2ufCg+taZAfj1GQO+iXZSNQ5NKxBqHdr8Z5AORdN3ywMZ3MP7802PB2fDtUEdMCWojww5+Tmjq7RR/ATHkzFKAMY/hkK+U2DJVxSWjyV1E/M41eokyp

In [39]:
# Print the final response
print(res["messages"][-1].content)

[{'type': 'text', 'text': 'Thought: The capital of India is well-known to be New Delhi.\nFinal Answer: The capital of India is New Delhi.', 'extras': {'signature': 'EuoDCucDAWkUfRNpQoAM5KtLXx0xzP+2o++LK5MytvxR45aPtP4Qk7tDSQgw3NVcd6rbHq6dVK+16hUCXoBRnldsYbE4WOpTupCqdRN3HUhARN0qna71pRty3HAKu+TS/T68Hp45H5Jpw8Vlo/nRK24JrAKEP85n+uqnZx2XvXDcOegzvTTL6ww0xmgDbDUCYx8Se5NyPZ2+CyCuBN2A1wjdQXKLvduifGt3YLzbDyY4zisMNpP1B7vGdbd24E/5dNq9ngOZOVN8fT9KUTbuZd+ZKr7bYoY97I8vZoR2tC2wKnDl90OBgK1vvezZ/ZRMs/kPfBBZWNBxWZhPPC5fR8zMb6kJDdxzckYSm9LLi4uINdoaENBOgHa3KCorkhspHj8c8Q99usE/7Nxs9nIlpekTbzsfk6APJzZ+FzdRUfVYwU/dqClHOfEV022ySW2IAfbBF3dEQCWFaq/mdDmELZcoseJHAaV+jMbk/0aYgV3mCfgSpXdWbgIH6i2ufCg+taZAfj1GQO+iXZSNQ5NKxBqHdr8Z5AORdN3ywMZ3MP7802PB2fDtUEdMCWojww5+Tmjq7RR/ATHkzFKAMY/hkK+U2DJVxSWjyV1E/M41eokypvEeud5hWMCI+rXR7bUoac3XRb2l0VYDPsrJ9w=='}}]
